# Offensive Production Returning, Leaving, Incoming
- based on code originally from the 2024-25 season book ('production_returning.ipynb' to clean and quantify the rosters and stats and 'preseason_production_plots_v1.02.ipynb' for plotting functions)

## Setup and Dependncies

In [3]:
# Dependencies

import os
import sys
import pandas as pd
import numpy as np


# path to TEMP folder
temp_folder = os.path.join(os.getcwd(), '..', 'TEMP')
# Data folder
data_folder = os.path.join(os.getcwd(), '..', 'data', 'player_info')
# Image folder
img_folder = os.path.join(os.getcwd(), '..', 'images')
# Logo folder
logo_folder = os.path.join(os.getcwd(), '..', 'images', 'logos')
# School folder
school_folder = os.path.join(os.getcwd(), '..', 'data', 'school_info')

## Path to 2024-25 stats (csv)
stats_file = os.path.join(data_folder, 'NCAA_stats_2024-25.csv')
stats_2024_df = pd.read_csv(stats_file)


################################################################################
roster_file = os.path.join(data_folder, 'EP_master_roster_v0.2_9-14.csv') # PATH TO THE ROSTER FILE -  Currently Using Elite Prospects Scrape
roster_file = os.path.join(data_folder, 'roster_2025_26_first_run_v4_ex20250827.csv') # PATH TO ROSTER FILE FROM CHN
roster_2025_df = pd.read_csv(roster_file) # read as dataframe

# Path to school info table (csv)
school_info_file = os.path.join(school_folder, 'arena_school_info.csv')
school_info_df = pd.read_csv(school_info_file)

In [4]:
## Display head of each table to check the load
# print(roster_2025_df.head())
# print(stats_2024_df.head())
# print(school_info_df.head())

In [5]:
roster_2025_df.head(35)

# Print roster length
print(f"Roster length: {len(roster_2025_df)}")

### HOTFIX - REMOVE GOALIES
roster_2025_df = roster_2025_df[roster_2025_df['Position'] != 'Goaltenders']

# Print updated roster length
print(f"Updated roster length: {len(roster_2025_df)}")

Roster length: 2046
Updated roster length: 1826


## Dictionaries and Constants
- Conference Membership Updated to Remove American Intl who dropped their program after 2024-25 season

In [6]:
## Conference Membership

atlantic = ['Air Force', 'Army', 'Bentley', 'Canisius', 'Holy Cross', 'Mercyhurst', 
            'Niagara', 'RIT', 'Robert Morris', 'Sacred Heart']

big_ten = ['Michigan', 'Michigan State', 'Minnesota', 'Notre Dame', 'Ohio State', 'Penn State', 'Wisconsin']

ccha = ['Augustana', 'Bemidji State', 'Bowling Green', 'Ferris State', 'Lake Superior', 'Michigan Tech', 
        'Minnesota State', 'Northern Michigan', 'St Thomas']

ecac = ['Brown', 'Clarkson', 'Colgate', 'Cornell', 'Dartmouth', 'Harvard', 'Princeton', 'Quinnipiac',
        'Rensselaer', 'St Lawrence', 'Union', 'Yale']

hockey_east = ['Boston College', 'Boston University', 'Connecticut', 'Maine', 'Massachusetts', 'Mass Lowell',
                'Merrimack', 'New Hampshire', 'Northeastern', 'Providence', 'Vermont']

nchc = ['Arizona State', 'Colorado College', 'Denver', 'Miami', 'Minnesota Duluth', 'North Dakota', 'Omaha', 'St Cloud State',
        'Western Michigan']

independents = ['Alaska Anchorage', 'Alaska', 'Lindenwood', 'Long Island', 'Stonehill']

# Create a dictionary of {Team: logo_abv} for each team with .png added to the end
logo_mapping = {}

for index, row in school_info_df.iterrows():
    logo_mapping[row['Team']] = row['logo_abv'] + '.png'

# print(logo_mapping)

#### Style and Colors

In [7]:
color_scheme = {
    'Returning Goals': '#2E8B57',  # Dark Green
    'Returning Assists': '#90EE90',  # Light Green
    'Departed Goals': '#B22222',  # Dark Red
    'Departed Assists': '#FF7F7F',  # Light Red
    'New Goals': '#1E90FF',  # Dark Blue
    'New Assists': '#87CEFA'  # Light Blue
}


### Data Cleaning and Modifications
- Remove Trouble character from Team Names


In [8]:


######
# Remove - from CHN Team names
stats_2024_df['Team'] = stats_2024_df['Team'].str.replace('-', ' ')
# roster_2024_df['Team'] = roster_2024_df['Team'].str.replace('-', ' ')
stats_2024_df['Team'] = stats_2024_df['Team'].str.replace('.', '')
stats_2024_df['Team'] = stats_2024_df['Team'].str.replace("'", '')
# Lower Case columns
stats_2024_df.columns = stats_2024_df.columns.str.lower()

# stats_2024_df.head()

#### Rename Columns in Roster File for easier merging
# Renam Current Team to Team
roster_2025_df = roster_2025_df.rename(columns={'Current Team': 'Team'})

# Create ne column called CLean_Player by merging First and Last Name
roster_2025_df['Clean_Player'] = roster_2025_df['First_Name'] + ' ' + roster_2025_df['Last_Name']
# Strip leading and trailing spaces
roster_2025_df['Clean_Player'] = roster_2025_df['Clean_Player'].str.strip()

# Lower Case columns
roster_2025_df.columns = roster_2025_df.columns.str.lower()

# roster_2025_df.head(35)

## Indentify Transfers

### CHN Data
- Compare 2025 roster to 2024 year end stats and flag players as returning vs departed
- Output 'transfers' dataframe with player, team 2024 and team 2025

In [10]:
# Step 1: Identify players that appear on different teams in the 2023 stats compared to the 2024 roster

# Merge the dataframes on the player name ('Clean_Player') to compare teams
transfer_df = pd.merge(stats_2024_df[['clean_player', 'team']], roster_2025_df[['clean_player', 'team']], on='clean_player', suffixes=('_2024', '_2025'))

# Filter for players who have a different team in 2023 compared to 2024
transfers = transfer_df[transfer_df['team_2024'] != transfer_df['team_2025']]
# # Rename Clean_Player to Player
transfers = transfers.rename(columns={'clean_player': 'player'})


# Reindex and save as csv to TEMP folder
transfers = transfers.reset_index(drop=True)
transfers.to_csv(os.path.join(temp_folder, 'transfers_2025.csv'))

# Create a count of transfers in per team
transfer_counts = transfers['team_2024'].value_counts().reset_index()
transfer_counts.columns = ['team', 'transfer_count']

## Print report on transfers


# transfers.head()
transfer_counts.head(10)


,team,transfer_count
0,Rensselaer,14
1,American Intl,10
2,Merrimack,7
3,Miami,6
4,Long Island,6
5,Omaha,6
6,Minnesota Duluth,5
7,Quinnipiac,5
8,Colorado College,5
9,Clarkson,4


## Other Transfer Data - The Rink Live
- Taken from the Rink Live Transfer Tracker, seemingly the best source available online